<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/hands_on_ml_with_scikit-learn_Aurelien_textbook/Chap_17_Autoencoders%2C_GANs%2C_and_Diffusion_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Autoencoders are artificial neural networks capable of learning dense representations of the input data, called latent representations or codings, without any supervision (i.e the training set is unlabeled). These codings typically have a much lower dimensionality than the input data, therefore making autoencoders useful for dimensionality reduction, especially for visualization purposes. They also act as feature detectors, and they can be used for unsupervised pretraining of deep neural networks. some autoencoders are also generative models capable of randomly generating new data that looks very similar to the training data.

Generative Adversarial networks(GANs) are also neural nets capable of generating data. They are also widely used for super resolution, colorization, powerful image editing, turning a simple sketch into a photorealistic image, predicting the next frames in a video, augmenting a dataset, generating other types of data, identifying the weakness in other models to strengthen them, and more.

### How Autoencoders, GANs and Diffusion models work:
1. Autoencoders simply learn to copy their inputs to their outputs. However, the autoencoder is constrained and thus, this prevents it form copying the inputs directly to the outputs hence forcing it to learn efficient ways of representing the data.

2. GANs are composed of two neural networks: a generator that tries to generate data that looks similar to the training data, and a discriminator that tries to tell the real data from fake data. The generator and the discriminator compete against each other during training: the generator is often compared to a criminal trying to make realistic counterfeit money, while the discriminator is like the police investigator trying to tell real money from the fake. hence the "adversarial" name.

3. Denoising Diffusion probabilistic Model(DDPM) is trained to remove a tiny bit of noise from an image. If you then take an image entirely full of Gaussian noise and repeatedly run the diffusion model on that image, a high quality image will gradually emerge, similar to the training images but not identical.

### Autoencoders
It is hard to memorize long sequences and that's what makes it useful to recognize patterns in data. An autoencoder looks at the inputs, converts them to an efficient latent representation, and then spits out something that(hopefully) looks very close to the inputs. It is composed of two parts: an encoder(or recognition network) that converts the inputs to a latent representation, followed by a decoder(or generative network) that converts the internal representations to the outputs. The outputs of an autoencoder are often called the reconstructions because the autoencoder tries to reconstruct the inputs. the cost function contains a reconstruction loss that penalizes the model when the reconstructions are different from the inputs. The internal representations have a lower dimensionality than the input data, hence the autoencoder is said to be undercomplete, An undercomplete autoencoder cannot trivially copy its inputs to the codings, yet it must find a way to output a copy of its inputs. Therefore it is forced to learn the most important features in the input data(and drop the unimportant ones)

### Performing PCA with an Undercomplete Linear AutoEncoder

In [ ]:
import tensorflow as tf

In [ ]:
# A simple linear autoencoder to perform PCA on a 3D dataset, projecting it to 2D
encoder = tf.keras.Sequential([
    tf.keras.layers.Dense(2)
])
decoder = tf.keras.Sequential([
    tf.keras.layers.Dense(3)
])
autoencoder = tf.keras.Sequential([encoder, decoder])
optimizer = tf.keras.optimizers.SGD(learning_rate=0.5)
autoencoder.compile(loss="mse", optimizer=optimizer)

# to perform pca, we do not use any activation fxn(i.e all neurons are linear),
# and the cost function is MSE. That is because pca is a linear transformation.

In [ ]:
history = autoencoder.fit(X_train, X_train, epochs=500, verbose=False)
# the train data is used as both input and target

NameError: ignored

#### Stacked Autoencoders
Adding more layers helps the autoencoder learn more complex codings, but one should be careful not to make the autoencoder too powerful else it just learns to map eaach input to a single arbitrary number(and the decoder learns the reverse mapping). Obviously, such an autoencoder will reconstruct the training data perfectly, but it will not have learned any useful data representation in the process, and it is unlikely to generalize well to new instances.

In [ ]:
stacked_encoder = tf.keras.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(30, activation="relu")
])
stacked_decoder = tf.keras.Sequential([
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(28*28),
    tf.keras.layers.Reshape([28, 28])
])
stacked_autoencoder = tf.keras.Sequential([
    stacked_encoder, stacked_decoder
])
stacked_autoencoder.compile(loss="mae", optimizer="nadam")
history = stacked_autoencoder.fit(X_train, X_train, epochs=20,
                                  validation_data=(X_valid, X_valid))

While autoencoders can be used for dimensionality reductions, if reducing the dimensionality is done for visualization purposes, they do not give great results compared to other dimensionality reduction algorithms. However, one of its advantage is that they can handle large datasets, with many instances and many features. A useful strategy is to use an autoencoder to reduce the dimensionality down to a reasonable level, then use another dimensionality reduction algorithm for visualization.

### Unsupervised Pretraining Using Stacked Autoencoders
If one has a large dataset but most of it is unlabeled, you can first train a stacked autoencoder using all the data, then reuse the lower layers to create a neural network for your actual task and train it using the labeled data. When training the classifier, if you really don't have much labeled training data, you may want to freeze the pretrained layers(atleast the lower ones).

### Denoising Autoencoders
One of the ways we can force the autoencoder to learn useful features is to add noise to its inputs, training it to recover the original, noise free inputs

### Generative Adversarial Networks
A GAN is composed of two neural networks:
1. Generator:
This takes a random distribution(typically gaussian) as input and outputs some data(typically an image).
2. Takes either a fake image from the generator or a real image from the training set as input, and must guess whether the input image is fake or real.

During training, the generator and the discriminator have opposite goals: the discriminator tries to tell fake images from real images, while the generator tries to produce images that look real enough to trick the discriminator. Because the GAN is composed of two networks with different objectives, it cannot be trained like a regular neural network. Each training iteration is divided into two phases:
1. In the first phase, we train the discriminator. A batch of real images is sampled from the training set and is completed with an equal number of fake images produced by the generator. The labels are set to 0 for fake images and 1 for real images, and the discriminator is trained on this labeled batch for one step. Backpropagation only optimizes the weights of the discriminator during this phase.
2. In the second phase, we train the generator. We first use it to produce another batch of fake images, and once again, the discriminator is used to tell whether the images are fake or real. This time, we do not add real images in the batch and all the labels are set to 1. In orther words, we want the generator to produce images that the discriminator will(wrongly) believe to be real. The weights of the discriminator are frozen during this step, so backpropagation only affects the weights of the generator.

The generator never actually sees any real images, yet it gradually learns to produce convincing fake images. All it gets is gradients flowing back through the discriminator.Fortunately, the better the discriminator gets, the more information about the real images is contained in these secondhand gradients so the generator can make significant progress.

In [ ]:
codings_size = 30
Dense = tf.keras.layers.Dense
generator = tf.keras.Sequential([
    Dense(100, activation="relu", kernel_initializer="he_normal"),
    Dense(150, activation="relu", kernel_initializer="he_normal"),
    Dense(28*28, activation="sigmoid"),
    tf.keras.layers.Reshape([28, 28])
])

discriminator = tf.keras.Sequential([
    tf.keras.layers.Flatten(),
    Dense(150, activation="relu", kernel_initializer="he_normal"),
    Dense(100, activation="relu", kernel_initializer="he_normal"),
    Dense(1, activation="sigmoid")
])
gan = tf.keras.Sequential([generator, discriminator])

In [ ]:
# The generator will only be trained through the gan model, so we don't need
# to compile it and the discriminator should not be trained during the second
# phase, so we make it non trainable before compiling the gan model
discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

The trainable attribute is taken into account by keras only when compiling a model, so after running the above code, the discriminator is trainable if we call its fit() method or its train_on_batch method while it is not trainable when we call these methods on the gan model.

Since the training loop is unusual, we cannot use the regular fit() method, instead, we will write a custom training loop.  To do this, we first need to create a dataset to iterate through the images.

In [ ]:
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(buffer_size=100)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

#### The Difficulties of Training GANs

During training, the generator and the discriminator constantly try to outsmart each other in a zero-sum game. As training advances, the game may end up in a state called Nash equilibrium: this is when no player would be better off changing their own strategy, assuming the other players do not change theirs.
The biggest difficulty is called mode collapse: this is when the generator's outputs gradually become less diverse. Suppose that the generator gets better at producing convincing shoes than any other class. It will fool the discriminator a bit more with shoes, and this will encourage it to produce even more images of shoes. Gradually, it will forget how to produce anything else. Meanwhile, the only fake images that the discriminator will see will be shoes so it will also forget how to discriminate fake images of other classes. Eventually, when the discriminator manages to discriminate fake shoes from the real ones, the generator will be forced to move to another class. It may then become good at shirts, forgetting about shoes and the discriminator will follow. The GAN may gradually cycle across a few classes, never really becoming very good at any of them.

Moreover, because the generator and the discriminator are constantly pushing against each other, their parameters may end up oscillating and becoming unstable. Training may begin properly, then suddenly diverge for no apparent reason, due to these instabilities. And since many factors affect these complex dynamics, GANs are very sensitive to the hyperparameters: you may have to spend a lot of effort fine tuning them.

Check the book to learn about some techniques that perform well, and to learn about diffusion models too.

In [ ]:
### The difficulties of trai